# Introduction

This project builds a movie recommendation system using collaborative filtering to suggest movies based on user preferences.
**bold text**

# IMPORT LIBRARIES

In [ ]:
# For data handling
import pandas as pd
import numpy as np

# For visualization (EDA)
import matplotlib.pyplot as plt
import seaborn as sns

# LOAD DATASETS

## Dataset Description

The dataset used in this project contains:
- User IDs
- Movie titles
- Ratings given by users

This data helps in understanding user preferences and building the recommendation system.

## Data Preprocessing

In this step:
- Datasets are loaded
- Data is cleaned (if needed)
- Columns are prepared for analysis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ratings = pd.read_csv("/content/drive/MyDrive/u.data", sep="\t",
                      names=["userId","movieId","rating","timestamp"])

movies = pd.read_csv("/content/drive/MyDrive/u.item", sep="|", encoding="latin-1",
                     names=["movieId","title","release_date","video_release_date",
                            "IMDb_URL","unknown","Action","Adventure","Animation",
                            "Children","Comedy","Crime","Documentary","Drama",
                            "Fantasy","Film-Noir","Horror","Musical","Mystery",
                            "Romance","Sci-Fi","Thriller","War","Western"])

# BASIC EDA

EDA is performed to understand the dataset better and extract useful insights such as:
- Most rated movies
- Highest rated movies
- Distribution of ratings

In [ ]:
ratings.head()

In [ ]:
movies.head()

In [ ]:
ratings.tail()

In [ ]:
movies.tail()

In [ ]:
# Check data info (columns, types)

ratings.info()

In [ ]:
movies.info()

In [ ]:
# check missing values
ratings.isnull().sum()

In [ ]:
movies.isnull().sum()

In [ ]:
ratings = pd.read_csv("/content/drive/MyDrive/u.data", sep="\t", names=["userId","movieId","rating","timestamp"])
movies = pd.read_csv("/content/drive/MyDrive/u.item", sep="|", encoding="latin-1", header=None)
movies = movies[[0,1]] # Select only movieId and title columns
movies.columns = ["movieId","title"] # Rename columns

df = pd.merge(ratings, movies, on="movieId")

In [ ]:
most_rated = df.groupby('title')['rating'].count().sort_values(ascending=False).head(10)
print(most_rated)

In [ ]:
# most rated movies in the dataset
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
most_rated.plot(kind='bar')
plt.title("Top 10 Most Rated Movies")
plt.xlabel("Movies")
plt.ylabel("Number of Ratings")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Which movies are rated highest by users
avg_rating = df.groupby('title')['rating'].mean().sort_values(ascending=False).head(10)
print(avg_rating)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
sns.histplot(df["rating"], bins=10, kde=True)
plt.title("Distribution of Movie Ratings")
plt.xlabel("Rating")
plt.ylabel("Count")
plt.show()

# MERGE DATASET

In [ ]:
# Merge ratings and movies on movieId
df = pd.merge(ratings, movies, on="movieId")


In [ ]:
# Keep only useful columns
df = df[["userId","movieId","rating","title"]]


In [ ]:
df

# Filter Popular Movies

In [ ]:
# Count number of ratings per movie
movie_ratings = df.groupby('title')['rating'].count()


In [ ]:
movie_ratings.head()

In [ ]:
# Select movies with more than 50 ratings
popular_movies = movie_ratings[movie_ratings > 50].index

In [ ]:
len(popular_movies)

In [ ]:
# Filter dataset
df = df[df['title'].isin(popular_movies)]

In [ ]:
df.head()

# Bar Graph for Top Popular Movies

In [ ]:
#top 10 most rated (popular) movies
top_movies = df.groupby("title")["rating"].count().sort_values(ascending=False).head(10)

In [ ]:
import matplotlib.pyplot as plt

top_movies.plot(kind='bar')

plt.title("Top 10 Most Popular Movies")
plt.xlabel("Movie Title")
plt.ylabel("Number of Ratings")

plt.xticks(rotation=90)
plt.show()

Create User-Movie Matrix
# Pivot Table Creation

A pivot table is created to transform the dataset into a matrix format.

- Rows represent users (userId)
- Columns represent movies (title)
- Values represent ratings given by users

This matrix helps in identifying similarities between movies based on user ratings.

In [ ]:
# convert data into matrix form
movie_matrix = df.pivot_table(index="userId",
                             columns="title",
                             values="rating")

movie_matrix = movie_matrix.fillna(0)
movie_matrix.head()

# Calculate SimilaritY

We calculate similarity between movies using correlation.

Movies with similar rating patterns among users will have higher similarity scores.

In [ ]:
# Compare movies with each other
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(movie_matrix.T)

similarity_df = pd.DataFrame(similarity,
                             index=movie_matrix.columns,
                             columns=movie_matrix.columns)

In [ ]:
# Take small sample (important, full matrix is too big)
sample = similarity_df.iloc[:10, :10]

plt.figure(figsize=(8,6))
sns.heatmap(sample, cmap='coolwarm')

plt.title("Movie Similarity Heatmap")
plt.xlabel("Movies")
plt.ylabel("Movies")

plt.show()

# RECOMMENDATION FUNCTION

In [ ]:
# “Take column → sort → skip first → print next 5”
def recommend(movie):
    similar = similarity_df[movie]
    similar = similar.sort_values(ascending=False)
    print(similar[1:6])

In [ ]:
recommend("You So Crazy (1994)")

VISUAL

In [ ]:
import matplotlib.pyplot as plt
def recommend_visual(movie):
    similar = similarity_df[movie].sort_values(ascending=False)[1:6]

    similar.plot(kind='bar')
    plt.title(f"Recommendations for {movie}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
recommend_visual("Star Wars (1977)")

# Add Error Handling

In [ ]:
# Error handling ensures program doesn’t give wrong input”
def recommend(movie):
    if movie not in similarity_df.columns:
        print("Movie not found!")
        return

    similar = similarity_df[movie].sort_values(ascending=False)
    print(similar[1:6])

In [ ]:
recommend("Fargo (1996)")

In [ ]:
recommend("ABC Movie")

# User Input

In [ ]:
# System recommends movies based on similar user preferences
movie_name = input("Enter movie name: ")
recommend(movie_name)

#Show Available Movies

In [ ]:
print(movie_matrix.columns.tolist()[:20])

In [ ]:
def recommend_visual(movie):
    if movie not in similarity_df.columns:
        print("Movie not found!")
        return

    similar = similarity_df[movie].sort_values(ascending=False)[1:6]

    plt.figure(figsize=(8,5))   # important
    similar.plot(kind='line')

    plt.title(f"Recommendations for {movie}")
    plt.xlabel("Movies")
    plt.ylabel("Similarity Score")

    plt.xticks(rotation=45)
    plt.show()

In [ ]:
recommend_visual("Wolf (1994)")

In [ ]:
recommend("Star Wars (1977)")
recommend("Toy Story (1995)")
recommend("12 Angry Men (1957)")

# Model Evaluation

Since this is a recommendation system, traditional accuracy metrics are not directly applicable.

The system is evaluated based on:
- Similarity between movies
- Relevance of recommended movies

Future improvements may include:
- RMSE (Root Mean Square Error)
- Precision

# Conclusion

The movie recommendation system successfully suggests movies based on user similarity using collaborative filtering.

However, it has some limitations:
- Cold start problem (new users or movies)
- Sparse data

Future improvements include hybrid recommendation systems and deep learning approaches.